# Phase 1: Foundation of Retrieval-Augmented Generation (RAG)

This notebook establishes the foundational infrastructure for an Enterprise-grade RAG system. The primary goal is to demonstrate a seamless integration between a Large Language Model (LLM) and a vector-based knowledge store using the **LangChain Expression Language (LCEL)**.

### Objectives:
* Implement environment authentication for High-Inference LLMs.
* Establish a localized vector database using **FAISS**.
* Construct a robust RAG pipeline for deterministic query resolution.

## 1. Environment Setup
We begin by installing the necessary dependencies to support high-performance embeddings and orchestration.

In [ ]:
# Silent installation to keep the environment clean
!pip install -qU \
    langchain \
    langchain-groq \
    langchain-community \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu

## 2. Component Initialization
Importing core modules required for the RAG architecture.

In [ ]:
import os
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## 3. Security & API Management
Configuring credentials for the Groq Inference Engine.

In [ ]:
# Securely handling API keys for production readiness
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

## 4. Knowledge Base Definition
Defining the "Ground Truth" data. For this foundational phase, we use a structured list representing corporate policy snippets.

In [ ]:
# Simulated Enterprise Knowledge Base
corporate_knowledge = [
    "Permanent employees are entitled to 12 days of annual leave per fiscal year.",
    "Overtime rates are applied at 150% of the standard hourly wage for the initial hour of additional work.",
    "Technical support for hardware-related issues is managed by the IT Department via internal extension 505.",
    "The corporate headquarters is situated in the central business district of Jakarta, Indonesia.",
    "Company health insurance coverage extends to the employee and up to four dependents."
]

def format_context(documents):
    """Helper function to format retrieved documents for the LLM prompt."""
    return "\n\n".join(doc.page_content for doc in documents)

## 5. Semantic Indexing with FAISS
Converting textual data into high-dimensional vector embeddings to enable efficient semantic search.

In [ ]:
# Initializing the HuggingFace embeddings model (all-MiniLM-L6-v2)
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Creating the vector store and initializing the retriever
vector_store = FAISS.from_texts(corporate_knowledge, embeddings_model)
knowledge_retriever = vector_store.as_retriever(search_kwargs={"k": 2})

## 6. Logic Orchestration (LCEL)
Constructing a modular RAG pipeline. This architecture ensures that the LLM is always provided with relevant context before generating a response.

In [ ]:
# Defining the System Instruction
rag_template = """
You are an AI Corporate Assistant. Answer the following question based ONLY on the provided context.
If the information is not present in the context, state that the answer is not available in the internal database.

Context:
{context}

Question: {question}

Response:
"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)

# Initializing Llama 3.3
inference_engine = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

# Final Pipeline Assembly
rag_system = (
    {"context": knowledge_retriever | format_context, "question": RunnablePassthrough()}
    | rag_prompt
    | inference_engine
    | StrOutputParser()
)

## 7. System Verification
Executing sample queries to validate the accuracy of the retrieval system.

In [ ]:
# Test Case: Annual Leave Policy
query_input = "Tell me about the annual leave policy for permanent employees."
print(f"User Inquiry: {query_input}")
print("-" * 50)
print(f"System Output:\n{rag_system.invoke(query_input)}")